# This starts n=x (4 in this example) mpi processes using iPython's ipyparallel module
<div class="alert alert-block alert-info">
    <b>Note:</b> This notebook uses ipyparallel and mpi4py for parallel distribution of string beads. It takes less than one minute to complete running interactively on 4-cores of an Intel 2.3 GHz processor MacBookPro using 4 parallel tasks and the parameters specified below.</div>

In [1]:
import ipyparallel as ipp
cluster = ipp.Cluster(engines="MPI",n=2)
cluster.start_and_connect_sync()

Starting 2 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/2 [00:00<?, ?engine/s]

# This script (realization of aladfy_mpi.py in jupyter):
## 1. Creates an initial alanine dipeptide structure and minimizes for all processes
## 2. Distributes the calculation of dihedral restrained minimization across the processes available
## Set-up call to pyCHARMM functional libraries. 
<div class="alert alert-block alert-info">
    <b>Note:</b> We use ipyparallel "magics" (<i>%%px</i>) to enable parallel execution of the cell blocks</div>

In [ ]:
%%px
import os
import sys
import subprocess
import numpy as np
import pandas as pd

from pycharmm import *
import pycharmm

# We will loop over the phi/psi angles to construct the phi/psi
# surface and distribute this across processes to reduce cost using mpi
# Add in mpi support
from mpi4py import MPI
comm = MPI.COMM_WORLD
nproc = comm.Get_size()
rank = comm.Get_rank()
# Logical flag to invoke gbmv energy surface
gbmv = False
# Function to plot surfaces
#
def pltFYMap():
    import pickle
    if gbmv:
        with open('gbmv/fymap-ala_gbmv.pkl', 'rb') as fh:
            fymap = pickle.load(fh)
    else:
        with open('vacuum/fymap-ala.pkl', 'rb') as fh:
            fymap = pickle.load(fh)

    en_df = pd.DataFrame.from_dict(fymap)
    en_df.sort_values(by=['F','Y'],inplace=True)
    fymap = en_df.to_dict(orient='list')
    F = np.linspace(-180,180,36)
    Y = F
    # reshape the energy array into nxn array
    # use transpose (.T) because of loop order above
    ener = np.reshape(np.asarray(fymap['ener']),(len(F),len(Y)))
    ener = ener.T
    emax = np.max(ener)
    emin = np.min(ener)
    erange = emax - emin
    ehigh = np.ceil(emin+1.1*erange)
    elow = np.floor(emax-1.1*erange)
    F,Y=np.meshgrid(F,Y)
    # Now let's plot the data f/y map
    from mpl_toolkits.mplot3d import axes3d
    import matplotlib.pyplot as plt
    from matplotlib import cm
    %matplotlib notebook
    fig = plt.figure()
    ax = plt.axes(projection='3d')
    plt.rcParams.update({'font.size':12})
    # Plot the 3D surface
    ax.contour3D(F,Y,ener,50,cmap='inferno')
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_zlim(elow, ehigh)
    
    if gbmv: ax.set_title(r'($\phi$, $\psi$) GBMV Surface for Alanine dipeptide')
    else: ax.set_title(r'($\phi$, $\psi$) Vacuum Surface for Alanine dipeptide')
    ax.set_xlabel(r'$\phi$')
    ax.set_ylabel(r'$\psi$')
    ax.set_zlabel('E (kcal/mol)')
    if gbmv: plt.savefig('gbmv/fysurf-ala_gbmv.pdf')    
    else: plt.savefig('vacuum/fysurf-ala.pdf')    
    plt.show()
    fig = plt.figure()
    left, bottom, width, height = 0.1, 0.1, 0.8, 0.8
    ax = fig.add_axes([left, bottom, width, height]) 
    cp = plt.contourf(F, Y, ener)
    plt.colorbar(cp) 
    if gbmv: ax.set_title(r'($\phi$, $\psi$) GBMV Surface for Alanine dipeptide')
    else: ax.set_title(r'($\phi$, $\psi$) Vacuum Surface for Alanine dipeptide')
    ax.set_xlabel(r'$\phi$')
    ax.set_ylabel(r'$\psi$')
    if gbmv: plt.savefig('gbmv/fycontour-ala_gbmv.pdf')
    else: plt.savefig('vacuum/fycontour-ala.pdf')
    plt.show()
    return
#####################################################################
###############PYCHARMM SCRIPTING STARTS HERE########################
ov = settings.set_verbosity(5)
settings.set_warn_level(-5)
print(f'Working from rank {rank}')
read.rtf('../toppar/top_all36_prot.rtf')
read.prm('../toppar/par_all36m_prot.prm')
read.sequence_string('ALA')  # ALAD for full dipeptide
generate.new_segment(seg_name='ALAD',
                first_patch='ACE', #comment to use alad residue
                last_patch='CT3',  #comment to use alad residue
                setup_ic=True)
ic.prm_fill(replace_all=True)
ic.seed(1,'CAY',1,'CY',1,'N')  
ic.build()
NonBondedScript(**{'cutnb': 16,
                            'ctofnb': 14,
                            'ctonnb': 12,
                            'atom': True,
                            'vatom': True,
                            'eps': 1,	
                            'switch': True,
                            'vswitch': True,
                            'cdie': True}).run()

minimize.run_abnr(**{'nstep': 1000,
                         'tolenr': 1e-3,
                         'tolgrd': 1e-3})
comm.barrier()
# template for f/y restraints
Fcons = '1 CY 1 N 1 CA 1 C'
Ycons = '1 N 1 CA 1 C 1 NT'
# set up phi/psi grid to apply restraints and
# compute energy
F = np.linspace(-180,180,36)
Y = F
fymap = {'F':[],
         'Y':[],
         'ener':[]}
for iphi,f in enumerate(F):
    if not ( iphi % nproc == rank ): continue
    for y in Y:
        # turn off noise
        # Need to use stream here because no api for cons dihe
        cons_methods.dihe(selection=Fcons,cldh=False,force=500,minimum=f'{f:4.2f}')
        cons_methods.dihe(selection=Ycons,cldh=False,force=500,minimum=f'{y:4.2f}')
        settings.set_verbosity(5)
        minimize.run_abnr(**{'nstep': 1000,
                           'tolenr': 1e-3,
                           'tolgrd': 1e-3})
        if gbmv:
            opl = settings.set_verbosity(0)
            read.stream(../toppar/radii_c36gbmvop.str)
            settings.set_verbosity(opl)
            script.CommandScript('gbmv', beta=-12, p3=0.65, watr=1.4, shift=-0.102,slope=0.9085,
                                 p6=8, sa=0.005, wtyp=2, nphi=38, cutnum=100, kappa=0, weight=True).run()
            settings.set_verbosity(0)
            settings.set_warn_level(-5)
            minimize.run_abnr(**{'nstep': 500,
                               'tolenr': 1e-3,
                               'tolgrd': 1e-3})

        cons_medhods.dihe(cldh=True)
        fymap['F'].append(f)
        fymap['Y'].append(y)
        fymap['ener'].append(energy.get_total())
        if gbmv: lingo.charmm_script('gbmv clear')
comm.barrier()
for r in range(1,nproc):
    if rank == r:
        req = comm.isend(fymap,dest=0,tag=10+r)
        req.wait()
    elif rank == 0:
        req = comm.irecv(source=r,tag=10+r)
        t = req.wait()
        for k in t.keys():
            for i in t[k]: fymap[k].append(i)
comm.barrier()
if rank == 0:
    import pickle
    if gbmv: 
        with open('gbmv/fymap-ala_gbmv.pkl', 'wb') as fh:
            pickle.dump(fymap, fh)
    else: 
        with open('vacuum/fymap-ala.pkl', 'wb') as fh:
            pickle.dump(fymap, fh)
    pltFYMap()
comm.barrier()

[stdout:1] Working from rank 1


[stdout:0] Working from rank 0
  
 CHARMM>     read rtf card -
 CHARMM>     name ../toppar/top_all36_prot.rtf
 VOPEN> Attempting to open::../TOPPAR/TOP_ALL36_PROT.RTF::
 MAINIO> Residue topology file being read from unit  91.
 TITLE> *>>>>>>>>CHARMM36 ALL-HYDROGEN TOPOLOGY FILE FOR PROTEINS <<<<<<
 TITLE> *>>>>> INCLUDES PHI, PSI CROSS TERM MAP (CMAP) CORRECTION <<<<<<<
 TITLE> *>>>>>>>>>>>>>>>>>>>>>>>>>> MAY 2011 <<<<<<<<<<<<<<<<<<<<<<<<<<<<
 TITLE> * ALL COMMENTS TO THE CHARMM WEB SITE: WWW.CHARMM.ORG
 TITLE> *             PARAMETER SET DISCUSSION FORUM
 TITLE> *


%px:   0%|          | 0/2 [00:00<?, ?tasks/s]